In [1]:
import pandas as pd
import numpy as np

def puhasta_maakond(nimi):
    """Ühtlustab maakondade nimed unikaalseks võtmetunnuseks."""
    if pd.isna(nimi):
        return None
    # Eemaldame jutumärgid ja üleliigsed tühikud tekstist
    nimi = str(nimi).strip().replace('"', '')
    if nimi.lower() in ["eesti", "kokku", "tallinn", "harju maakond, v.a tallinn"]:
        return None  
    return nimi

def impordi_pa21(file_path):
    """Loeb sisse perioodi tabeli PA21 (2005-2017) puhtast CSV formaadist."""
    # Kasutame utf-8-sig kodeeringut, et neutraliseerida peidetud failialguse sümbolid (BOM)
    df = pd.read_csv(file_path, skiprows=1, encoding="utf-8-sig")
    
    # Puhastame veergude nimed
    df.columns = df.columns.str.replace('"', '').str.strip()
    
    df["Aasta"] = df["Aasta"].astype(str).str.replace('"', '').str.strip()
    df["Kvartal"] = df["Kvartal"].astype(str).str.replace('"', '').str.strip()
    
    # Loome ühtse kvartali tähise (nt "2005 I")
    df["quarter"] = df["Aasta"] + " " + df["Kvartal"].str.replace(" kvartal", "")
    
    # Määrame maakonna tulba positsiooni järgi (tabelis kolmas tulp ehk indeks 2)
    df["county"] = df.iloc[:, 2].apply(puhasta_maakond)
    
    # Puhastame palganumbri (tabelis neljas tulp ehk indeks 3)
    df["salary"] = pd.to_numeric(df.iloc[:, 3], errors='coerce')
    
    return df.dropna(subset=["county", "salary"])[["quarter", "county", "salary"]]

def impordi_maatriks_tabel(file_path):
    """Üldine funktsioon maatrikskujul tabelite (PA004 ja PA117) laadimiseks positsioonipõhiselt."""
    df = pd.read_csv(file_path, skiprows=1, encoding="utf-8-sig")
    
    # Puhastame veergude nimed jutumärkidest ja tühikutest
    df.columns = df.columns.str.replace('"', '').str.strip()
    
    # POSITSIOONIPÕHINE TUVASTUS: Maakond/Haldusüksus on alati teine tulp (indeks 1)
    df["county"] = df.iloc[:, 1].apply(puhasta_maakond)
    df = df.dropna(subset=["county"])
    
    # Valime ainult kvartalitega veerud (kõik veerud alates indeksist 2, mis sisaldavad sõna "kvartal")
    kvartali_veerud = [col for col in df.columns if "kvartal" in col.lower()]
    
    # Keerame maatriksi pikaks tabeliks (Unpivot / Melt)
    df_melted = df.melt(id_vars=["county"], value_vars=kvartali_veerud, var_name="Raw_Quarter", value_name="salary")
    
    # Normaliseerime kvartali kuju ("2021 I kvartal" -> "2021 I")
    df_melted["quarter"] = df_melted["Raw_Quarter"].str.replace('"', '').str.replace(" kvartal", "").str.strip()
    df_melted["salary"] = pd.to_numeric(df_melted["salary"], errors='coerce')
    
    return df_melted.dropna(subset=["salary"])[["quarter", "county", "salary"]]

# === PEAPROGRAMM / LIITMINE ===
def loo_pikk_palga_aegrida(fail_pa21, fail_pa004, fail_pa117):
    print("Mootor käivitatud: uute andmefailide sisselugemine positsioonide alusel...")
    
    df1 = impordi_pa21(fail_pa21)
    df2 = impordi_maatriks_tabel(fail_pa004)
    df3 = impordi_maatriks_tabel(fail_pa117)
    
    # Liidame kõik kolm perioodi kokku ühise aegrea alla
    pikk_aegrida = pd.concat([df1, df2, df3], ignore_index=True)
    
    # Eemaldame duplikaadid (kui mõni kvartal peaks tabelite vahel kattuma)
    pikk_aegrida = pikk_aegrida.drop_duplicates(subset=["quarter", "county"])
    
    # Sorteerime tulemuse loogiliselt ajaliselt ja maakonniti
    pikk_aegrida = pikk_aegrida.sort_values(by=["county", "quarter"]).reset_index(drop=True)
    
    return pikk_aegrida

# Käivitamise näide:
df_palk_2005_2025 = loo_pikk_palga_aegrida(
    fail_pa21="PA21_20260922-104007_raw.csv", 
    fail_pa004="PA004_20260922-103201_raw.csv", 
    fail_pa117="PA117_20260930-094438_raw.csv"
)

# Kuvame tulemuse kontrolliks
print("--- AEGREAD VALMIS (Esimene ots) ---")
print(df_palk_2005_2025.head(15))
print("\n--- AEGREAD VALMIS (Viimane ots) ---")
print(df_palk_2005_2025.tail(15))

Mootor käivitatud: uute andmefailide sisselugemine positsioonide alusel...
--- AEGREAD VALMIS (Esimene ots) ---
     quarter         county  salary
0     2005 I  Harju maakond     552
1    2005 II  Harju maakond     611
2   2005 III  Harju maakond     574
3    2005 IV  Harju maakond     637
4     2006 I  Harju maakond     639
5    2006 II  Harju maakond     696
6   2006 III  Harju maakond     670
7    2006 IV  Harju maakond     746
8     2007 I  Harju maakond     766
9    2007 II  Harju maakond     831
10  2007 III  Harju maakond     790
11   2007 IV  Harju maakond     882
12    2008 I  Harju maakond     897
13   2008 II  Harju maakond     947
14  2008 III  Harju maakond     891

--- AEGREAD VALMIS (Viimane ots) ---
       quarter        county  salary
1275   2022 IV  Võru maakond    1348
1276    2023 I  Võru maakond    1322
1277   2023 II  Võru maakond    1468
1278  2023 III  Võru maakond    1410
1279   2023 IV  Võru maakond    1492
1280    2024 I  Võru maakond    1446
1281   2024 II 

In [2]:
df_palk_2005_2025.to_csv("Eesti_keskmine_palk_maakonniti_2005_2025.csv", index=False, encoding="utf-8-sig")